# Feature Engineering part 1 - Feature Generation

This notebook aggregates historical data into an initial set of features. Details about the tables
being aggregated can be found here: https://www.kaggle.com/competitions/home-credit-default-risk/data.

In [1]:
PROJECT_DIR = ".."
DATA_DIR = f"{PROJECT_DIR}/data"
ARTIFACTS_DIR = PROJECT_DIR

#### Initializations

In [7]:
import home_credit_risk.data_util as data_util
import home_credit_risk.features as ft

feature_generator = ft.FeatureGenerator(
    data_dir=DATA_DIR,
    column_abbrev_file_path=f"{PROJECT_DIR}/config/column_abbreviations.csv"
)

## Table: credit_card_balance

In [8]:
feature_generator.generate_features(
    table_name="credit_card_balance",
    time_boundary_column="MONTHS_BALANCE",
    time_boundaries=(-36, -24, -12, -6, -3),
    cross_aggregate_all=True,
)


Generating features for credit_card_balance...
Loading data from /content/drive//MyDrive/home_credit_risk/data/credit_card_balance.csv...
Aggregating full table
Checking for features to eliminate...
Eliminating 1/7 features...
Checking for features to eliminate...
Eliminating 371/560 features...
Checking for features to eliminate...
Eliminating 2/80 features...
credit_card_balance: 273 features generated. Merging...
Aggregating data for MONTHS_BALANCE >= -36
Checking for features to eliminate...
Eliminating 3/7 features...
Checking for features to eliminate...
Eliminating 415/560 features...
Checking for features to eliminate...
credit_card_balance: 229 features generated. Merging...
Aggregating data for MONTHS_BALANCE >= -24
Checking for features to eliminate...
Eliminating 3/7 features...
Checking for features to eliminate...
Eliminating 415/560 features...
Checking for features to eliminate...
credit_card_balance: 229 features generated. Merging...
Aggregating data for MONTHS_BALAN

## Table: POS_CASH_balance

In [9]:
feature_generator.generate_features(
    table_name="POS_CASH_balance",
    time_boundary_column="MONTHS_BALANCE",
    time_boundaries=(-36, -24, -12, -6, -3),
    cross_aggregate_all=True,
)


Generating features for POS_CASH_balance...
Loading data from /content/drive//MyDrive/home_credit_risk/data/POS_CASH_balance.csv...
Aggregating full table
Checking for features to eliminate...
Eliminating 2/9 features...
Checking for features to eliminate...
Eliminating 99/180 features...
Checking for features to eliminate...
POS_CASH_balance: 108 features generated. Merging...
Aggregating data for MONTHS_BALANCE >= -36
Checking for features to eliminate...
Eliminating 2/9 features...
Checking for features to eliminate...
Eliminating 107/180 features...
Checking for features to eliminate...
POS_CASH_balance: 100 features generated. Merging...
Aggregating data for MONTHS_BALANCE >= -24
Checking for features to eliminate...
Eliminating 2/9 features...
Checking for features to eliminate...
Eliminating 123/180 features...
Checking for features to eliminate...
POS_CASH_balance: 84 features generated. Merging...
Aggregating data for MONTHS_BALANCE >= -12
Checking for features to eliminate..

## Table: bureau

In [10]:
feature_generator.generate_features(
    table_name="bureau",
    time_boundary_column="DAYS_CREDIT",
    time_boundaries=(-1080, -720, -360, -180, -90),
    cross_aggregate_all=True,
)


Generating features for bureau...
Loading data from /content/drive//MyDrive/home_credit_risk/data/bureau.csv...
Aggregating full table
Checking for features to eliminate...
Checking for features to eliminate...
Eliminating 68/192 features...
Checking for features to eliminate...
Eliminating 1/4 features...
Checking for features to eliminate...
Eliminating 144/192 features...
Checking for features to eliminate...
Eliminating 4/15 features...
Checking for features to eliminate...
Eliminating 497/720 features...
Checking for features to eliminate...
bureau: 461 features generated. Merging...
Aggregating data for DAYS_CREDIT >= -1080
Checking for features to eliminate...
Eliminating 1/4 features...
Checking for features to eliminate...
Eliminating 100/192 features...
Checking for features to eliminate...
Eliminating 1/4 features...
Checking for features to eliminate...
Eliminating 144/192 features...
Checking for features to eliminate...
Eliminating 6/15 features...
Checking for features 

## Table: bureau_balance

In [11]:
import os
import pandas as pd

def preprocess_bureau_balance(raw_data: pd.DataFrame) -> pd.DataFrame:
    parent_ids_df = pd.read_csv(os.path.join(DATA_DIR, "bureau.csv"),
                                usecols=[data_util.APP_ID_COL, "SK_ID_BUREAU"])
    parent_ids_df = parent_ids_df.drop_duplicates()
    parent_ids_df.set_index("SK_ID_BUREAU", inplace=True)
    return raw_data.join(parent_ids_df, how="inner")

feature_generator.generate_features(
    table_name="bureau_balance",
    cross_aggregate_all=True,
    preprocessor=preprocess_bureau_balance
)


Generating features for bureau_balance...
Loading data from /content/drive//MyDrive/home_credit_risk/data/bureau_balance.csv...
Preprocessing...
Aggregating full table
Checking for features to eliminate...
Checking for features to eliminate...
Eliminating 16/32 features...
Checking for features to eliminate...
bureau_balance: 28 features generated. Merging...


## Table: installments_payments

In [12]:
import numpy as np

def preprocess_installment_payments(raw_data: pd.DataFrame) -> pd.DataFrame:
    tolerance = -5

    raw_data["DAY_PMNT_DIFF"] = np.where(
        raw_data["DAYS_ENTRY_PAYMENT"].isnull(),
        np.where(
            raw_data["DAYS_INSTALMENT"] >= tolerance,
            np.nan,
            -raw_data["DAYS_INSTALMENT"],
        ),
        raw_data["DAYS_ENTRY_PAYMENT"] - raw_data["DAYS_INSTALMENT"],
    )

    late_pay_ranges = [
        raw_data["DAY_PMNT_DIFF"] > 30,
        raw_data["DAY_PMNT_DIFF"] > 0
    ]
    late_pay_labels = ["gt_30", "1_to_30"]
    raw_data["DAYS_LATE"] = np.select(late_pay_ranges, late_pay_labels, default="not_late")
    raw_data["DAYS_LATE"] = raw_data["DAYS_LATE"].astype('category')

    return raw_data

def postprocess_installment_payments(aggregated_data: pd.DataFrame, feature_prefix: str):
    not_late = aggregated_data[ft.FeatureNamer.join(feature_prefix, "DAYS_LATE", "not_late")]
    gt_30 = aggregated_data[ft.FeatureNamer.join(feature_prefix, "DAYS_LATE", "gt_30")]
    lte_30 = aggregated_data[ft.FeatureNamer.join(feature_prefix, "DAYS_LATE", "1_to_30")]
    count_sum = not_late + gt_30 + lte_30
    aggregated_data[ft.FeatureNamer.join(feature_prefix, "DAYS_LATE_RATIO", "not_late")] = not_late / count_sum
    aggregated_data[ft.FeatureNamer.join(feature_prefix, "DAYS_LATE_RATIO", "1_to_30")] = lte_30 / count_sum

feature_generator.generate_features(
    table_name="installments_payments",
    time_boundary_column="DAYS_INSTALMENT",
    time_boundaries=(-1080, -720, -360, -180, -90),
    cross_aggregate_all=True,
    preprocessor=preprocess_installment_payments,
    postprocessor=postprocess_installment_payments
)


Generating features for installments_payments...
Loading data from /content/drive//MyDrive/home_credit_risk/data/installments_payments.csv...
Preprocessing...
Aggregating full table
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Postprocessing...
installments_payments: 117 features generated. Merging...
Aggregating data for DAYS_INSTALMENT >= -1080
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Postprocessing...
installments_payments: 117 features generated. Merging...
Aggregating data for DAYS_INSTALMENT >= -720
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Postprocessing...
installments_payments: 117 features generated. Merging...
Aggregating data for DAYS_INSTALMENT >= -360
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate.

## Table: previous_application

In [13]:
def preprocess_previous_application(raw_data: pd.DataFrame) -> pd.DataFrame:
    original_length = len(raw_data)
    raw_data = raw_data[(raw_data["FLAG_LAST_APPL_PER_CONTRACT"] == "Y") & (raw_data["NFLAG_LAST_APPL_IN_DAY"] == 1)]
    print(f"Removed {original_length - len(raw_data)} rows from previous_application table according to the \"last application\" flags")

    columns_to_drop = [
        "FLAG_LAST_APPL_PER_CONTRACT",
        "NFLAG_LAST_APPL_IN_DAY",
        "SK_ID_PREV"
    ]
    return raw_data.drop(columns=columns_to_drop)

feature_generator.generate_features(
    table_name="previous_application",
    time_boundary_column="DAYS_DECISION",
    time_boundaries=(-1080, -720, -360, -180, -90),
    cross_aggregate_all=False,
    cross_aggregate_categories={"NAME_CONTRACT_STATUS": ["Refused"]},
    preprocessor=preprocess_previous_application
)


Generating features for previous_application...
Loading data from /content/drive//MyDrive/home_credit_risk/data/previous_application.csv...
Preprocessing...
Removed 9261 rows from previous_application table according to the "last application" flags
Aggregating full table
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Eliminating 28/68 features...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Eliminating 2/28 features...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features to eliminate...
Checking for features 

## Save Generated Features



In [16]:
feature_generator.save_generated_features("generated_features_all")

Saving generated features, dataset shapes: train=(307511, 6300), test=(48744, 6299)
Files saved:
/content/drive//MyDrive/home_credit_risk/data/generated_features_all_info.txt
/content/drive//MyDrive/home_credit_risk/data/generated_features_all_train.parquet
/content/drive//MyDrive/home_credit_risk/data/generated_features_all_test.parquet
